# FrameSat-AI: Kaggle Experiment Controller
This notebook serves as the primary experiment controller for FrameSat-AI on Kaggle. You can customize hyperparameter overrides in **Step 1**, initialize the orchestrator in **Step 2**, and execute the full end-to-end pipeline in **Step 3**.

### Step 1: Define Experiment Configuration Overrides
Edit the dictionary below to override default settings (e.g. `epochs`, `batch_size`, `learning_rate`, `train_resize`, `channel`, `experiment_name`).

In [ ]:
# ==========================================
# Reusable Experiment Presets (Copy/Paste or assign to CONFIG_OVERRIDES)
# ==========================================

FAST_DEBUG = {
    "epochs": 2,
    "batch_size": 2,
    "train_resize": [256, 256]
}

FULL_TRAINING = {
    "epochs": 100,
    "batch_size": 8,
    "train_resize": [512, 512]
}

PAPER_RESULTS = {
    "epochs": 150,
    "batch_size": 8,
    "train_resize": [512, 512],
    "save_predictions": True
}

# ==========================================
# FrameSat-AI Experiment Configuration Overrides
# ==========================================

CONFIG_OVERRIDES = {
    # Experiment Identification
    "experiment_name": "GOES19_RIFE_v1",

    # Training Hyperparameters
    "epochs": 10,
    "batch_size": 4,
    "learning_rate": 1e-4,
    "optimizer": "AdamW",          # Supported: SGD, Adam, AdamW
    "weight_decay": 1e-4,
    "scheduler": "cosine",          # Supported: cosine, reduce_on_plateau, none
    "mixed_precision": True,       # Enables PyTorch AMP (autocast)

    # Dataset Parameters
    "channel": 13,
    "train_resize": [384, 384],

    # Scientific Evaluation
    "num_eval_events": 10,
    "save_predictions": True,

    # Reproducibility
    "seed": 42
}

### Step 2: Bootstrap Orchestrator
Discover the project workspace path and load `KaggleOrchestrator`.

In [ ]:
import os
import sys

# Dynamic discovery of bundle path
bundle_path = None
for root, dirs, files in os.walk("/kaggle/input"):
    if "training" in dirs and "evaluation" in dirs:
        bundle_path = root
        break
if not bundle_path:
    if os.path.exists("/kaggle/working/training/train.py"):
        bundle_path = "/kaggle/working"

if not bundle_path:
    raise RuntimeError("Could not find FrameSat-AI source code bundle.")

if bundle_path not in sys.path:
    sys.path.insert(0, bundle_path)

from framesat_kaggle.kaggle_orchestrator import KaggleOrchestrator

orchestrator = KaggleOrchestrator()
orchestrator.bundle_path = bundle_path
print(f"\033[32m✔\033[0m Bootstrapped KaggleOrchestrator from: {bundle_path}")

### Step 3: Run Full End-to-End Pipeline
Execute environment setup, path discovery, pre-flight validation, config resolution with overrides, training, post-training evaluation, and output packaging.

In [ ]:
orchestrator.run(overrides=CONFIG_OVERRIDES)